# Multimodal Audio-Visual Deepfake Detection
## Kaggle Notebook

This notebook implements a **multimodal audio-visual deepfake detector** that combines:
- 🎥 **Visual Branch**: DINOv2 Vision Transformer for spatial features
- ⏱️ **Temporal Branch**: Transformer encoder for temporal dynamics
- 🔊 **Audio Branch**: wav2vec2 for audio forensics
- 🌊 **Frequency Branch**: Wavelet decomposition for forensic artifacts
- 🔗 **Fusion Module**: Cross-attention mechanism to combine all modalities

### Dataset
Designed for **Celeb-DF v2** (or any dataset with `real/` and `fake/` video folders):
- Input dataset should be mounted at `/kaggle/input/celebdf-v2/` (adjust path in the config cell)

### Quick Start
1. Run **Cell 1** – install dependencies  
2. Run **Cell 2** – imports  
3. Run **Cell 3** – configure paths & hyper-parameters  
4. Run **Cells 4-9** – paste model & utility code into the session  
5. Run **Cell 10** – prepare dataset metadata  
6. Run **Cell 11** – train the model  
7. Run **Cell 12** – run inference / evaluation  


## 1. Install Dependencies

In [ ]:
%%capture
# Kaggle already ships PyTorch with the correct CUDA version for its GPU.
# The line below is left commented out to avoid overwriting it; uncomment only
# if you need a specific PyTorch build (adjust the CUDA suffix, e.g. cu118/cu121).
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q

!pip install timm>=0.9.0 -q
!pip install transformers>=4.37.0 accelerate>=0.26.0 -q

# Video / audio processing
!pip install opencv-python-headless>=4.9.0 librosa>=0.10.0 soundfile>=0.12.0 av>=11.0.0 -q

# Wavelet analysis
!pip install PyWavelets>=1.5.0 -q

# Visualisation & utilities
!pip install matplotlib seaborn tqdm pyyaml einops -q

print("\u2705 All packages installed")


## 2. Imports

In [ ]:
import os, sys, json, math, time, random, shutil
from pathlib import Path
from typing import Dict, Optional, Tuple, List

import numpy as np
import cv2
import librosa
import pywt
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from tqdm.notebook import tqdm

import warnings
warnings.filterwarnings('ignore')

print(f"Python  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()} | device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


## 3. Configuration

Edit the paths and hyper-parameters below to match your Kaggle dataset and compute budget.

* `DATA_ROOT` – directory that contains **`real/`** and **`fake/`** sub-folders with `.mp4` videos.
* Reduce `NUM_FRAMES`, `FRAME_SIZE`, and `BATCH_SIZE` if you run out of GPU memory.


In [ ]:
# ── Kaggle / data paths ──────────────────────────────────────────────────────
DATA_ROOT        = "/kaggle/input/celebdf-v2"   # ← adjust to your dataset path
WORKING_DIR      = "/kaggle/working"
CHECKPOINT_DIR   = os.path.join(WORKING_DIR, "checkpoints")
LOG_DIR          = os.path.join(WORKING_DIR, "logs")
METADATA_FILE    = os.path.join(WORKING_DIR, "metadata.json")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

# ── Configuration dictionary (mirrors configs/config.yaml) ───────────────────
CONFIG = {
    "project_name"     : "deepfake_detection",
    "experiment_name"  : "multimodal_av_detector_kaggle",
    "seed"             : 42,

    "data": {
        "raw_dir"          : DATA_ROOT,
        "processed_dir"    : WORKING_DIR,
        "video_extensions" : [".mp4", ".avi", ".mov", ".mkv"],
        "audio_sample_rate": 16000,
        "video_fps"        : 30,
        "frame_size"       : 224,   # reduce to 112 / 196 to save memory
        "num_frames"       : 16,    # reduce to 8 to save memory
        "clip_duration"    : 2.0,
    },

    "model": {
        "name": "MultimodalAVDetector",
        "visual": {
            "backbone"        : "dinov2_vits14",  # vits14=384-d, vitb14=768-d
            "pretrained"      : True,
            "freeze_backbone" : False,
            "embed_dim"       : 384,
        },
        "temporal": {
            "type"       : "transformer",
            "num_layers" : 4,
            "num_heads"  : 6,
            "dropout"    : 0.1,
        },
        "audio": {
            "backbone"        : "wav2vec2",
            "pretrained"      : True,
            "freeze_backbone" : False,
            "embed_dim"       : 768,
        },
        "frequency": {
            "wavelet_type": "db8",
            "levels"      : 3,
        },
        "fusion": {
            "type"      : "cross_attention",
            "hidden_dim": 512,
            "num_heads" : 8,
            "dropout"   : 0.15,
        },
        "classifier": {
            "hidden_dims": [256, 128],
            "dropout"    : 0.2,
            "num_classes": 2,
        },
    },

    "training": {
        "batch_size"    : 4,      # reduce to 2 if OOM
        "num_epochs"    : 30,
        "learning_rate" : 1e-4,
        "weight_decay"  : 1e-4,
        "optimizer"     : "adamw",
        "scheduler"     : "cosine",
        "warmup_epochs" : 5,
        "loss": {
            "type"            : "cross_entropy",
            "label_smoothing" : 0.1,
        },
        "augmentation": {
            "horizontal_flip"       : True,
            "color_jitter"          : True,
            "gaussian_blur"         : True,
            "compression_emulation" : True,
            "compression_quality"   : [70, 95],
        },
    },

    "validation": {
        "batch_size"     : 8,
        "split_ratio"    : 0.15,
        "eval_frequency" : 1,
    },

    "testing": {
        "batch_size" : 8,
        "tta"        : False,
        "threshold"  : 0.5,
    },

    "explainability": {
        "enabled"           : True,
        "methods"           : ["gradcam"],
        "save_visualizations": True,
        "visualization_dir" : os.path.join(LOG_DIR, "visualizations"),
    },

    "logging": {
        "use_wandb"       : False,
        "log_frequency"   : 10,
        "checkpoint_dir"  : CHECKPOINT_DIR,
        "save_frequency"  : 5,
        "keep_last_n"     : 3,
    },

    "hardware": {
        "device"           : "cuda",
        "num_workers"      : 2,
        "pin_memory"       : True,
        "mixed_precision"  : True,
        "compile_model"    : False,
    },

    "paths": {
        "pretrained_models": "models/pretrained",
        "outputs"          : os.path.join(WORKING_DIR, "outputs"),
        "logs"             : LOG_DIR,
    },
}

print("Configuration ready ✅")
print(f"  Data root  : {CONFIG['data']['raw_dir']}")
print(f"  Checkpoints: {CONFIG['logging']['checkpoint_dir']}")
print(f"  Logs       : {CONFIG['paths']['logs']}")


## 4. Utility Helpers

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Helpers (from utils/helpers.py)
# ─────────────────────────────────────────────────────────────────────────────

def set_seed(seed: int):
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_device(config: Dict) -> torch.device:
    """Return the best available compute device."""
    device_name = config['hardware']['device']
    if device_name == 'cuda' and torch.cuda.is_available():
        device = torch.device('cuda')
        print(f"Using CUDA: {torch.cuda.get_device_name(0)}")
    else:
        device = torch.device('cpu')
        print("Using CPU")
    return device


def count_parameters(model: nn.Module) -> int:
    """Count trainable parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def save_checkpoint(model, optimizer, epoch, loss, save_path, **kwargs):
    """Save model checkpoint."""
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
    torch.save({
        'epoch'               : epoch,
        'model_state_dict'    : model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss'                : loss,
        **kwargs
    }, save_path)
    print(f"Checkpoint saved → {save_path}")


def load_checkpoint(model, checkpoint_path, optimizer=None, device=None):
    """Load model checkpoint."""
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    if optimizer is not None and 'optimizer_state_dict' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    print(f"Checkpoint loaded ← {checkpoint_path}  (epoch {checkpoint.get('epoch', '?')})")
    return checkpoint


class AverageMeter:
    """Track running average."""
    def __init__(self):
        self.reset()
    def reset(self):
        self.val = self.avg = self.sum = self.count = 0
    def update(self, val, n=1):
        self.val   = val
        self.sum  += val * n
        self.count += n
        self.avg   = self.sum / self.count


def accuracy(output: torch.Tensor, target: torch.Tensor) -> float:
    with torch.no_grad():
        pred = output.argmax(dim=1)
        return (pred == target).sum().item() / target.size(0)


print("Utility helpers defined ✅")


## 5. Data Augmentation

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Augmentation (from utils/augmentation.py)
# ─────────────────────────────────────────────────────────────────────────────

class VideoAugmentation:
    """Video augmentation pipeline including compression emulation."""

    def __init__(self, horizontal_flip=True, color_jitter=True,
                 gaussian_blur=True, compression_emulation=True,
                 compression_quality_range=(70, 95), mode='train'):
        self.horizontal_flip       = horizontal_flip and (mode == 'train')
        self.color_jitter          = color_jitter and (mode == 'train')
        self.gaussian_blur         = gaussian_blur and (mode == 'train')
        self.compression_emulation = compression_emulation and (mode == 'train')
        self.compression_quality_range = compression_quality_range
        self.mode = mode

    def __call__(self, frames: np.ndarray) -> np.ndarray:
        if self.mode != 'train':
            return frames
        if self.horizontal_flip and random.random() > 0.5:
            frames = frames[:, :, :, ::-1].copy()
        if self.color_jitter and random.random() > 0.5:
            frames = self._apply_color_jitter(frames)
        if self.gaussian_blur and random.random() > 0.5:
            frames = self._apply_gaussian_blur(frames)
        if self.compression_emulation and random.random() > 0.5:
            frames = self._apply_compression(frames)
        return frames

    def _apply_color_jitter(self, frames):
        bf = random.uniform(0.8, 1.2)
        frames = np.clip(frames * bf, 0, 1)
        cf = random.uniform(0.8, 1.2)
        mean = frames.mean(axis=(2, 3), keepdims=True)
        frames = np.clip((frames - mean) * cf + mean, 0, 1)
        sf = random.uniform(0.8, 1.2)
        gray = frames.mean(axis=1, keepdims=True)
        frames = np.clip((frames - gray) * sf + gray, 0, 1)
        return frames

    def _apply_gaussian_blur(self, frames):
        ks = random.choice([3, 5])
        sigma = random.uniform(0.1, 2.0)
        out = []
        for t in range(frames.shape[0]):
            f = frames[t].transpose(1, 2, 0)
            f = cv2.GaussianBlur(f, (ks, ks), sigma)
            out.append(f.transpose(2, 0, 1))
        return np.stack(out)

    def _apply_compression(self, frames):
        quality = random.randint(*self.compression_quality_range)
        out = []
        for t in range(frames.shape[0]):
            f = (frames[t].transpose(1, 2, 0) * 255).astype(np.uint8)
            _, enc = cv2.imencode('.jpg', f, [int(cv2.IMWRITE_JPEG_QUALITY), quality])
            f = cv2.imdecode(enc, cv2.IMREAD_COLOR).astype(np.float32) / 255.0
            out.append(f.transpose(2, 0, 1))
        return np.stack(out)


class AudioAugmentation:
    """Audio augmentation pipeline."""

    def __init__(self, add_noise=True, mode='train'):
        self.add_noise = add_noise and (mode == 'train')
        self.mode = mode

    def __call__(self, audio: np.ndarray) -> np.ndarray:
        if self.mode != 'train':
            return audio
        if self.add_noise and random.random() > 0.5:
            noise_factor = random.uniform(0.001, 0.01)
            audio = np.clip(audio + np.random.randn(len(audio)) * noise_factor, -1.0, 1.0)
        return audio


def get_transforms(config: dict, mode: str = 'train'):
    """Create augmentation transforms from config."""
    aug = config['training']['augmentation']
    video_transform = VideoAugmentation(
        horizontal_flip=aug['horizontal_flip'],
        color_jitter=aug['color_jitter'],
        gaussian_blur=aug['gaussian_blur'],
        compression_emulation=aug['compression_emulation'],
        compression_quality_range=tuple(aug['compression_quality']),
        mode=mode,
    )
    audio_transform = AudioAugmentation(add_noise=True, mode=mode)
    return video_transform, audio_transform


print("Augmentation classes defined ✅")


## 6. Dataset

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Dataset (from utils/dataset.py)
# ─────────────────────────────────────────────────────────────────────────────

class DeepfakeDataset(Dataset):
    """Dataset for deepfake video detection."""

    def __init__(self, data_dir, metadata_file, num_frames=16, frame_size=224,
                 audio_sample_rate=16000, clip_duration=2.0,
                 transform=None, audio_transform=None, mode='train'):
        self.data_dir          = data_dir
        self.num_frames        = num_frames
        self.frame_size        = frame_size
        self.audio_sample_rate = audio_sample_rate
        self.clip_duration     = clip_duration
        self.transform         = transform
        self.audio_transform   = audio_transform
        self.mode              = mode
        self._warn_count       = 0

        with open(metadata_file) as f:
            meta = json.load(f)
        self.samples = meta[mode]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample     = self.samples[idx]
        video_path = os.path.join(self.data_dir, sample['path'])
        label      = sample['label']

        frames = self._load_video(video_path)
        audio  = self._load_audio(video_path)

        if self.transform      is not None: frames = self.transform(frames)
        if self.audio_transform is not None: audio  = self.audio_transform(audio)

        return {
            'video'     : torch.from_numpy(frames).float(),
            'audio'     : torch.from_numpy(audio).float(),
            'label'     : torch.tensor(label, dtype=torch.long),
            'video_path': video_path,
        }

    def _load_video(self, video_path):
        cap          = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if total_frames <= self.num_frames:
            idxs = list(range(total_frames))
            while len(idxs) < self.num_frames:
                idxs.append(idxs[-1])
        else:
            idxs = np.linspace(0, total_frames - 1, self.num_frames, dtype=int)

        frames = []
        for i in idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, i)
            ret, frame = cap.read()
            if ret:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = cv2.resize(frame, (self.frame_size, self.frame_size))
                frame = frame.astype(np.float32) / 255.0
                frames.append(frame)
            else:
                frames.append(frames[-1].copy() if frames else
                              np.zeros((self.frame_size, self.frame_size, 3), np.float32))
        cap.release()

        frames = np.stack(frames).transpose(0, 3, 1, 2)  # [T, C, H, W]
        return frames

    def _load_audio(self, video_path):
        target = int(self.audio_sample_rate * self.clip_duration)
        try:
            audio, _ = librosa.load(video_path, sr=self.audio_sample_rate,
                                    duration=self.clip_duration, mono=True)
            if len(audio) < target:
                audio = np.pad(audio, (0, target - len(audio)))
            else:
                audio = audio[:target]
        except Exception:
            if self._warn_count < 3:
                print(f"[Dataset] Audio unavailable for a sample in mode={self.mode}, using silence.")
                self._warn_count += 1
            audio = np.zeros(target, dtype=np.float32)
        return audio


def create_metadata_json(data_dir: str, output_file: str,
                         train_ratio=0.70, val_ratio=0.15):
    """
    Build metadata.json from a folder with real/ and fake/ sub-dirs.
    """
    samples = []
    for label, sub in enumerate(['real', 'fake']):
        sub_dir = os.path.join(data_dir, sub)
        if not os.path.isdir(sub_dir):
            print(f"[Warning] Sub-directory not found: {sub_dir}")
            continue
        for fname in os.listdir(sub_dir):
            if any(fname.lower().endswith(ext) for ext in ['.mp4', '.avi', '.mov', '.mkv']):
                samples.append({'path': os.path.join(sub, fname), 'label': label})

    random.shuffle(samples)
    n        = len(samples)
    t_end    = int(n * train_ratio)
    v_end    = t_end + int(n * val_ratio)
    metadata = {
        'train': samples[:t_end],
        'val'  : samples[t_end:v_end],
        'test' : samples[v_end:],
    }

    with open(output_file, 'w') as f:
        json.dump(metadata, f, indent=2)

    print(f"Metadata created → {output_file}")
    print(f"  train={len(metadata['train'])}  val={len(metadata['val'])}  test={len(metadata['test'])}")
    return metadata


print("Dataset class defined ✅")


## 7. Model Architecture

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Positional Encoding
# ─────────────────────────────────────────────────────────────────────────────

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=100):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        position  = torch.arange(max_len).unsqueeze(1)
        div_term  = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe        = torch.zeros(1, max_len, d_model)
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# ─────────────────────────────────────────────────────────────────────────────
# Visual Encoder (DINOv2)
# ─────────────────────────────────────────────────────────────────────────────

class VisualEncoder(nn.Module):
    """Visual feature extractor using DINOv2."""

    _DIM_MAP = {
        'dinov2_vits14': 384,
        'dinov2_vitb14': 768,
        'dinov2_vitl14': 1024,
        'dinov2_vitg14': 1536,
    }

    def __init__(self, backbone='dinov2_vits14', pretrained=True,
                 freeze_backbone=False, embed_dim=384):
        super().__init__()
        self.backbone_name = backbone
        self.embed_dim     = embed_dim

        if pretrained:
            try:
                self.backbone = torch.hub.load('facebookresearch/dinov2', backbone)
            except Exception as e:
                print(f"[VisualEncoder] Could not load pretrained {backbone}: {e}")
                print("[VisualEncoder] Falling back to random-init ViT via timm.")
                from timm import create_model
                self.backbone = create_model('vit_small_patch14_dinov2', pretrained=False)
        else:
            from timm import create_model
            self.backbone = create_model('vit_small_patch14_dinov2', pretrained=False)

        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        self.projection = nn.Linear(self._DIM_MAP.get(backbone, 384), embed_dim)

    def forward(self, x):
        with torch.set_grad_enabled(self.training):
            features = self.backbone(x)
        return self.projection(features)


# ─────────────────────────────────────────────────────────────────────────────
# Temporal Encoder
# ─────────────────────────────────────────────────────────────────────────────

class TemporalEncoder(nn.Module):
    """Transformer-based temporal modelling."""

    def __init__(self, embed_dim=384, num_layers=4, num_heads=6, dropout=0.1, max_seq_len=100):
        super().__init__()
        self.pos_encoding  = PositionalEncoding(embed_dim, dropout, max_seq_len)
        enc_layer          = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim * 4,
            dropout=dropout, activation='gelu', batch_first=True)
        self.transformer   = nn.TransformerEncoder(enc_layer, num_layers)
        self.temporal_pool = nn.AdaptiveAvgPool1d(1)

    def forward(self, x):          # x: [B, T, D]
        x = self.pos_encoding(x)
        x = self.transformer(x)
        return self.temporal_pool(x.transpose(1, 2)).squeeze(-1)  # [B, D]


# ─────────────────────────────────────────────────────────────────────────────
# Audio Encoder (wav2vec2)
# ─────────────────────────────────────────────────────────────────────────────

class AudioEncoder(nn.Module):
    """Audio feature extractor using wav2vec2."""

    def __init__(self, backbone='wav2vec2', pretrained=True,
                 freeze_backbone=False, embed_dim=768):
        super().__init__()
        from transformers import Wav2Vec2Model, Wav2Vec2Config
        if pretrained:
            try:
                self.backbone = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base-960h')
            except Exception as e:
                print(f"[AudioEncoder] Could not load pretrained wav2vec2: {e}")
                self.backbone = Wav2Vec2Model(Wav2Vec2Config())
        else:
            self.backbone = Wav2Vec2Model(Wav2Vec2Config())

        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        self.temporal_pool = nn.AdaptiveAvgPool1d(1)
        self.projection    = nn.Linear(self.backbone.config.hidden_size, embed_dim)

    def forward(self, audio):      # audio: [B, L]
        with torch.set_grad_enabled(self.training):
            h = self.backbone(audio).last_hidden_state  # [B, T, D]
        h = self.temporal_pool(h.transpose(1, 2)).squeeze(-1)  # [B, D]
        return self.projection(h)


# ─────────────────────────────────────────────────────────────────────────────
# Frequency Analyzer (wavelet decomposition)
# ─────────────────────────────────────────────────────────────────────────────

class FrequencyAnalyzer(nn.Module):
    """Wavelet-based forensic artifact detector."""

    def __init__(self, wavelet_type='db8', levels=3, input_channels=3):
        super().__init__()
        self.wavelet_type  = wavelet_type
        self.levels        = levels
        self.input_channels= input_channels
        self.feature_dim   = input_channels * (3 * levels + 1)
        self.band_weights  = nn.Parameter(torch.ones(self.feature_dim))
        self.feature_net   = nn.Sequential(
            nn.Linear(self.feature_dim, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, 128),              nn.ReLU())

    def _wavelet_decompose(self, frame):  # frame: [C, H, W] tensor
        frame_np = frame.cpu().numpy()
        features = []
        for c in range(self.input_channels):
            coeffs = pywt.wavedec2(frame_np[c], self.wavelet_type, level=self.levels)
            features.append(float(np.sqrt(np.mean(coeffs[0] ** 2))))
            for lc in coeffs[1:]:
                for sb in lc:
                    features.append(float(np.sqrt(np.mean(sb ** 2))))
        return torch.tensor(features, dtype=frame.dtype, device=frame.device)

    def forward(self, video):  # [B, T, C, H, W]
        B, T = video.shape[:2]
        all_f = []
        sample_idxs = torch.linspace(0, T - 1, min(8, T), dtype=torch.long)
        for b in range(B):
            ff = torch.stack([self._wavelet_decompose(video[b, t]) for t in sample_idxs])
            all_f.append(ff.mean(0))
        features = torch.stack(all_f) * self.band_weights
        return self.feature_net(features)  # [B, 128]


# ─────────────────────────────────────────────────────────────────────────────
# Fusion Module
# ─────────────────────────────────────────────────────────────────────────────

class FusionModule(nn.Module):
    """Cross-attention multimodal fusion."""

    def __init__(self, visual_dim=384, audio_dim=768, freq_dim=128,
                 hidden_dim=512, num_heads=8, dropout=0.15):
        super().__init__()
        self.visual_proj  = nn.Linear(visual_dim, hidden_dim)
        self.audio_proj   = nn.Linear(audio_dim,  hidden_dim)
        self.freq_proj    = nn.Linear(freq_dim,   hidden_dim)
        self.cross_attn   = nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True)
        self.self_attn    = nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True)
        self.ffn          = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 4, hidden_dim), nn.Dropout(dropout))
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.norm3 = nn.LayerNorm(hidden_dim)
        self.mod_attn = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 3),              nn.Softmax(dim=-1))

    def forward(self, visual, audio, frequency):
        v = self.visual_proj(visual).unsqueeze(1)   # [B,1,D]
        a = self.audio_proj(audio).unsqueeze(1)
        f = self.freq_proj(frequency).unsqueeze(1)

        v_attn, _ = self.cross_attn(v, a, a)
        v         = self.norm1(v + v_attn)

        all_f     = torch.cat([v, a, f], dim=1)     # [B,3,D]
        sa, _     = self.self_attn(all_f, all_f, all_f)
        all_f     = self.norm2(all_f + sa)
        all_f     = self.norm3(all_f + self.ffn(all_f))

        weights   = self.mod_attn(all_f.view(all_f.size(0), -1)).unsqueeze(-1)  # [B,3,1]
        return (all_f * weights).sum(dim=1)          # [B, D]


# ─────────────────────────────────────────────────────────────────────────────
# Classifier Head
# ─────────────────────────────────────────────────────────────────────────────

class Classifier(nn.Module):
    def __init__(self, input_dim=512, hidden_dims=[256, 128], num_classes=2, dropout=0.2):
        super().__init__()
        layers, in_d = [], input_dim
        for h in hidden_dims:
            layers += [nn.Linear(in_d, h), nn.ReLU(), nn.Dropout(dropout)]
            in_d = h
        layers.append(nn.Linear(in_d, num_classes))
        self.classifier = nn.Sequential(*layers)

    def forward(self, x):
        return self.classifier(x)


# ─────────────────────────────────────────────────────────────────────────────
# Full Model
# ─────────────────────────────────────────────────────────────────────────────

class MultimodalAVDetector(nn.Module):
    """
    Multimodal Audio-Visual Deepfake Detector.

    Forward input:
        video : [B, T, C, H, W]
        audio : [B, audio_length]
    Output dict:
        logits   : [B, num_classes]
        features : dict of intermediate tensors (when return_features=True)
    """

    def __init__(self, config: Dict):
        super().__init__()
        mc = config['model']
        self.visual_encoder   = VisualEncoder(**mc['visual'])
        self.temporal_encoder = TemporalEncoder(
            embed_dim=mc['visual']['embed_dim'],
            num_layers=mc['temporal']['num_layers'],
            num_heads=mc['temporal']['num_heads'],
            dropout=mc['temporal']['dropout'])
        self.audio_encoder    = AudioEncoder(**mc['audio'])
        self.frequency_analyzer = FrequencyAnalyzer(
            wavelet_type=mc['frequency']['wavelet_type'],
            levels=mc['frequency']['levels'])
        self.fusion           = FusionModule(
            visual_dim=mc['visual']['embed_dim'],
            audio_dim=mc['audio']['embed_dim'],
            hidden_dim=mc['fusion']['hidden_dim'],
            num_heads=mc['fusion']['num_heads'],
            dropout=mc['fusion']['dropout'])
        self.classifier       = Classifier(
            input_dim=mc['fusion']['hidden_dim'],
            **mc['classifier'])

    def forward(self, video, audio, return_features=False):
        B, T = video.shape[:2]
        vf   = self.visual_encoder(video.view(-1, *video.shape[2:]))   # [B*T, D]
        vf   = vf.view(B, T, -1)                                        # [B, T, D]
        tf   = self.temporal_encoder(vf)                                # [B, D]
        af   = self.audio_encoder(audio)                                # [B, D]
        ff   = self.frequency_analyzer(video)                           # [B, 128]
        fused= self.fusion(tf, af, ff)                                  # [B, hidden]
        logits = self.classifier(fused)                                 # [B, C]

        out = {'logits': logits}
        if return_features:
            out['features'] = dict(visual=vf, temporal=tf, audio=af, frequency=ff, fused=fused)
        return out

    def predict(self, video, audio):
        with torch.no_grad():
            out   = self.forward(video, audio)
            probs = F.softmax(out['logits'], dim=1)
            return probs.argmax(dim=1), probs


print("All model classes defined ✅")


## 8. Data Preparation

Run this cell to scan your dataset directory and create `metadata.json`.

> **Expected layout** inside `DATA_ROOT`:
> ```
> DATA_ROOT/
> ├── real/   ← real videos (.mp4 / .avi / .mov / .mkv)
> └── fake/   ← deepfake videos
> ```
>
> For **Celeb-DF v2**, first run `prepare_celebdf.py` to organise the raw download,
> or manually copy/symlink `Celeb-real/` + `YouTube-real/` → `real/`
> and `Celeb-synthesis/` → `fake/`.


In [ ]:
set_seed(CONFIG['seed'])

# Create / refresh metadata
metadata = create_metadata_json(
    data_dir    = CONFIG['data']['raw_dir'],
    output_file = METADATA_FILE,
    train_ratio = 1.0 - CONFIG['validation']['split_ratio'] * 2,
    val_ratio   = CONFIG['validation']['split_ratio'],
)

# Quick sanity check: load one sample
if metadata['train']:
    sample_path = os.path.join(CONFIG['data']['raw_dir'], metadata['train'][0]['path'])
    print(f"\nSample video path : {sample_path}")
    print(f"Exists            : {os.path.exists(sample_path)}")
else:
    print("\n⚠️  No training samples found. Check DATA_ROOT path.")


## 9. Training

In [ ]:
class Trainer:
    """End-to-end training loop for the deepfake detector."""

    def __init__(self, config: Dict, resume_path: str = None, num_epochs: int = None):
        self.config = config
        if num_epochs is not None:
            self.config['training']['num_epochs'] = int(num_epochs)

        set_seed(config['seed'])
        self.device = get_device(config)

        os.makedirs(config['logging']['checkpoint_dir'], exist_ok=True)
        os.makedirs(config['paths']['logs'],             exist_ok=True)

        # Model
        self.model = MultimodalAVDetector(config).to(self.device)
        print(f"Model → {count_parameters(self.model):,} trainable params")

        self._init_datasets()
        self._init_optimizer()
        self._init_loss()

        self.use_amp = config['hardware']['mixed_precision'] and self.device.type == 'cuda'
        self.scaler  = GradScaler('cuda') if self.use_amp else None
        self.start_epoch  = 0
        self.best_val_acc = 0.0

        if resume_path:
            ckpt = load_checkpoint(self.model, resume_path,
                                   optimizer=self.optimizer, device=self.device)
            self.start_epoch  = ckpt.get('epoch', -1) + 1
            self.best_val_acc = ckpt.get('val_acc', 0.0)
            print(f"Resuming from epoch {self.start_epoch + 1}  (best val acc so far: {self.best_val_acc:.4f})")

    def _init_datasets(self):
        tr, va = get_transforms(self.config, 'train'), get_transforms(self.config, 'val')
        kw = dict(
            data_dir          = self.config['data']['raw_dir'],
            metadata_file     = METADATA_FILE,
            num_frames        = self.config['data']['num_frames'],
            frame_size        = self.config['data']['frame_size'],
            audio_sample_rate = self.config['data']['audio_sample_rate'],
            clip_duration     = self.config['data']['clip_duration'],
        )
        self.train_dataset = DeepfakeDataset(**kw, transform=tr[0], audio_transform=tr[1], mode='train')
        self.val_dataset   = DeepfakeDataset(**kw, transform=va[0], audio_transform=va[1], mode='val')

        lkw = dict(num_workers=self.config['hardware']['num_workers'],
                   pin_memory=self.config['hardware']['pin_memory'])
        self.train_loader = DataLoader(self.train_dataset,
                                       batch_size=self.config['training']['batch_size'],
                                       shuffle=True, **lkw)
        self.val_loader   = DataLoader(self.val_dataset,
                                       batch_size=self.config['validation']['batch_size'],
                                       shuffle=False, **lkw)
        print(f"Train: {len(self.train_dataset)} | Val: {len(self.val_dataset)}")

    def _init_optimizer(self):
        name = self.config['training']['optimizer'].lower()
        lr   = self.config['training']['learning_rate']
        wd   = self.config['training']['weight_decay']
        if name == 'adam':
            self.optimizer = optim.Adam(self.model.parameters(), lr=lr, weight_decay=wd)
        elif name == 'adamw':
            self.optimizer = optim.AdamW(self.model.parameters(), lr=lr, weight_decay=wd)
        elif name == 'sgd':
            self.optimizer = optim.SGD(self.model.parameters(), lr=lr, momentum=0.9, weight_decay=wd)
        else:
            raise ValueError(f"Unknown optimizer: {name}")

        sched = self.config['training']['scheduler'].lower()
        n_ep  = self.config['training']['num_epochs']
        if sched == 'cosine':
            self.scheduler = optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=n_ep)
        elif sched == 'step':
            self.scheduler = optim.lr_scheduler.StepLR(self.optimizer, step_size=max(1, n_ep // 3), gamma=0.1)
        elif sched == 'plateau':
            self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(self.optimizer, mode='max', factor=0.5, patience=5)
        else:
            raise ValueError(f"Unknown scheduler: {sched}")

    def _init_loss(self):
        ls = self.config['training']['loss']['label_smoothing']
        self.criterion = nn.CrossEntropyLoss(label_smoothing=ls)

    def train_epoch(self, epoch):
        self.model.train()
        losses, accs = AverageMeter(), AverageMeter()
        pbar = tqdm(self.train_loader,
                    desc=f"Epoch {epoch+1}/{self.config['training']['num_epochs']}")
        for batch in pbar:
            video  = batch['video'].to(self.device)
            audio  = batch['audio'].to(self.device)
            labels = batch['label'].to(self.device)
            B      = video.size(0)

            if self.use_amp:
                with autocast(device_type='cuda'):
                    out  = self.model(video, audio)
                    loss = self.criterion(out['logits'], labels)
                self.optimizer.zero_grad()
                self.scaler.scale(loss).backward()
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                out  = self.model(video, audio)
                loss = self.criterion(out['logits'], labels)
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()

            losses.update(loss.item(), B)
            accs.update(accuracy(out['logits'], labels), B)
            pbar.set_postfix(loss=f'{losses.avg:.4f}', acc=f'{accs.avg:.4f}')

        return losses.avg, accs.avg

    @torch.no_grad()
    def validate(self):
        self.model.eval()
        losses, accs = AverageMeter(), AverageMeter()
        for batch in tqdm(self.val_loader, desc="Validating", leave=False):
            video  = batch['video'].to(self.device)
            audio  = batch['audio'].to(self.device)
            labels = batch['label'].to(self.device)
            out    = self.model(video, audio)
            loss   = self.criterion(out['logits'], labels)
            losses.update(loss.item(), video.size(0))
            accs.update(accuracy(out['logits'], labels), video.size(0))
        return losses.avg, accs.avg

    def train(self):
        n_ep = self.config['training']['num_epochs']
        print(f"\nStarting training for {n_ep} epochs")
        print("=" * 60)

        history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

        for epoch in range(self.start_epoch, n_ep):
            t0 = time.time()
            tr_loss, tr_acc = self.train_epoch(epoch)

            val_loss = val_acc = None
            if (epoch + 1) % self.config['validation']['eval_frequency'] == 0:
                val_loss, val_acc = self.validate()
                print(f"Epoch {epoch+1:3d}/{n_ep} | "
                      f"train loss={tr_loss:.4f} acc={tr_acc:.4f} | "
                      f"val loss={val_loss:.4f} acc={val_acc:.4f} | "
                      f"time={time.time()-t0:.1f}s")

                history['train_loss'].append(tr_loss)
                history['train_acc'].append(tr_acc)
                history['val_loss'].append(val_loss)
                history['val_acc'].append(val_acc)

                if isinstance(self.scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                    self.scheduler.step(val_acc)
                else:
                    self.scheduler.step()

                save_freq = self.config['logging']['save_frequency']
                if (epoch + 1) % save_freq == 0:
                    save_checkpoint(self.model, self.optimizer, epoch, val_loss,
                                    os.path.join(self.config['logging']['checkpoint_dir'],
                                                 f'ckpt_epoch_{epoch+1}.pth'),
                                    val_acc=val_acc, train_acc=tr_acc)

                if val_acc > self.best_val_acc:
                    self.best_val_acc = val_acc
                    save_checkpoint(self.model, self.optimizer, epoch, val_loss,
                                    os.path.join(self.config['logging']['checkpoint_dir'], 'best_model.pth'),
                                    val_acc=val_acc, train_acc=tr_acc)
                    print(f"  ✅ New best model saved  (val acc={val_acc:.4f})")

            print("=" * 60)

        print(f"\nTraining complete. Best val acc: {self.best_val_acc:.4f}")
        return history


print("Trainer class defined ✅")


In [ ]:
# ── Launch training ───────────────────────────────────────────────────────────
# Set resume_path to a checkpoint file if you want to continue a previous run.
trainer = Trainer(CONFIG, resume_path=None)
history = trainer.train()


## 10. Training Curves

In [ ]:
if history['train_loss']:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history['train_loss'], label='Train Loss')
    axes[0].plot(history['val_loss'],   label='Val Loss')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True)

    axes[1].plot(history['train_acc'], label='Train Acc')
    axes[1].plot(history['val_acc'],   label='Val Acc')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(True)

    plt.suptitle('Training History', fontsize=14)
    plt.tight_layout()
    fig_path = os.path.join(LOG_DIR, 'training_curves.png')
    plt.savefig(fig_path, bbox_inches='tight')
    plt.show()
    print(f"Figure saved → {fig_path}")
else:
    print("No training history to plot yet.")


## 11. Inference on a Single Video

In [ ]:
@torch.no_grad()
def predict_video(video_path: str, model: nn.Module, config: Dict, device: torch.device) -> Dict:
    """
    Run deepfake inference on a single video file.

    Args:
        video_path : absolute path to the video file
        model      : trained MultimodalAVDetector
        config     : CONFIG dictionary
        device     : torch device

    Returns:
        dict with keys: prediction, confidence, probabilities
    """
    model.eval()

    # Build a temporary one-sample dataset
    tmp_meta = os.path.join(WORKING_DIR, '_tmp_infer_meta.json')
    with open(tmp_meta, 'w') as f:
        json.dump({'test': [{'path': video_path, 'label': 0}]}, f)

    _, val_tf = get_transforms(config, 'val')
    ds = DeepfakeDataset(
        data_dir='',          # paths in metadata are absolute
        metadata_file=tmp_meta,
        num_frames=config['data']['num_frames'],
        frame_size=config['data']['frame_size'],
        audio_sample_rate=config['data']['audio_sample_rate'],
        clip_duration=config['data']['clip_duration'],
        transform=val_tf[0], audio_transform=val_tf[1],
        mode='test',
    )
    os.remove(tmp_meta)

    sample = ds[0]
    video  = sample['video'].unsqueeze(0).to(device)
    audio  = sample['audio'].unsqueeze(0).to(device)

    out    = model(video, audio)
    probs  = F.softmax(out['logits'], dim=1)[0]
    pred   = probs.argmax().item()

    return {
        'video_path'   : video_path,
        'prediction'   : 'Fake' if pred == 1 else 'Real',
        'confidence'   : probs[pred].item(),
        'probabilities': {'real': probs[0].item(), 'fake': probs[1].item()},
    }


# ── Example usage ─────────────────────────────────────────────────────────────
# Replace with the actual path to a video file you want to analyse.
# VIDEO_TO_TEST = "/kaggle/input/celebdf-v2/real/sample.mp4"

# model = trainer.model   # use the model from the training cell, OR:
# ckpt_path = os.path.join(CHECKPOINT_DIR, 'best_model.pth')
# load_checkpoint(model, ckpt_path, device=trainer.device)

# result = predict_video(VIDEO_TO_TEST, model, CONFIG, trainer.device)
# print(f"Prediction : {result['prediction']}")
# print(f"Confidence : {result['confidence']:.2%}")
# print(f"Real prob  : {result['probabilities']['real']:.2%}")
# print(f"Fake prob  : {result['probabilities']['fake']:.2%}")

print("predict_video() helper defined ✅")
print("Uncomment the example usage lines above to test on a specific video.")


## 12. Test-Set Evaluation

In [ ]:
@torch.no_grad()
def evaluate_test_set(model: nn.Module, config: Dict, device: torch.device):
    """Evaluate model on the held-out test split."""
    model.eval()
    _, val_tf = get_transforms(config, 'val')
    ds = DeepfakeDataset(
        data_dir=config['data']['raw_dir'],
        metadata_file=METADATA_FILE,
        num_frames=config['data']['num_frames'],
        frame_size=config['data']['frame_size'],
        audio_sample_rate=config['data']['audio_sample_rate'],
        clip_duration=config['data']['clip_duration'],
        transform=val_tf[0], audio_transform=val_tf[1],
        mode='test',
    )
    loader = DataLoader(ds, batch_size=config['testing']['batch_size'],
                        shuffle=False, num_workers=config['hardware']['num_workers'])

    all_preds, all_labels = [], []
    for batch in tqdm(loader, desc='Test evaluation'):
        out    = model(batch['video'].to(device), batch['audio'].to(device))
        preds  = out['logits'].argmax(dim=1).cpu()
        all_preds.append(preds)
        all_labels.append(batch['label'])

    all_preds  = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)

    test_acc = (all_preds == all_labels).float().mean().item()
    print(f"Test accuracy: {test_acc:.4f}  ({int((all_preds == all_labels).sum())}/{len(all_labels)})")

    # Per-class breakdown
    for cls, name in enumerate(['Real', 'Fake']):
        mask = all_labels == cls
        if mask.sum() > 0:
            acc = (all_preds[mask] == all_labels[mask]).float().mean().item()
            print(f"  {name}: {acc:.4f}  (n={mask.sum().item()})")

    return test_acc, all_preds, all_labels


# ── Run on test set ───────────────────────────────────────────────────────────
# Uncomment after training is complete:
# test_acc, preds, labels = evaluate_test_set(trainer.model, CONFIG, trainer.device)
print("evaluate_test_set() helper defined ✅")
print("Uncomment the last line to run evaluation on the test split.")
